# WindOps AI Copilot — Agent Decisions

This notebook demonstrates the LangGraph agent in action.

The agent:
1. Inspects the priority ranking to identify critical turbines
2. Retrieves detailed subscore breakdowns per turbine
3. Diagnoses the most likely fault based on signal patterns
4. Submits a structured, operational action plan for each turbine

All reasoning steps are captured and displayed for full traceability.

> **Prerequisite:** set `ANTHROPIC_API_KEY` in a `.env` file at the project root.

In [2]:
# ===============================
# SETUP
# ===============================

import sys
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sys.path.append("..")

from src.data_generation import load_demo_scenario
from src.features import build_features
from src.anomaly import run_anomaly_pipeline
from src.risk import run_risk_pipeline
from src.impact import run_impact_pipeline
from src.prioritization import run_prioritization_pipeline
from app.agent import run_agent

SCENARIO = "mixed"
TOP_N = 3
PLOT_STYLE = "seaborn-v0_8-whitegrid"
plt.style.use(PLOT_STYLE)

/usr/local/python/3.12.1/lib/python3.12/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 1. Pipeline

Run the full analytics pipeline to produce the priority ranking that the agent will consume.

In [3]:
# ===============================
# RUN FULL PIPELINE
# ===============================

print(f"Loading scenario: {SCENARIO}")
df_hourly   = load_demo_scenario(SCENARIO)
df_features = build_features(df_hourly)
df_anomaly, iso_model = run_anomaly_pipeline(df_features)
df_risk, risk_summary = run_risk_pipeline(df_anomaly)
loss_summary = run_impact_pipeline(df_risk)
priority     = run_prioritization_pipeline(risk_summary, loss_summary)

fault_map = df_hourly.groupby("turbine_id")["fault_type"].first().to_dict()

print(f"Turbines : {df_hourly['turbine_id'].nunique()}")
print(f"Hours    : {df_hourly['timestamp'].nunique()}")
print(f"\nTop {TOP_N} by priority:")
cols = ["priority_rank", "turbine_id", "priority_score", "risk_score_mean", "loss_mwh_total"]
print(priority[cols].head(TOP_N).to_string(index=False))

Loading scenario: mixed
Turbines : 20
Hours    : 720

Top 3 by priority:
 priority_rank turbine_id  priority_score  risk_score_mean  loss_mwh_total
             1     WTG-02        0.965385         0.621562      284.465325
             2     WTG-07        0.236127         0.153033      117.904157
             3     WTG-08        0.172303         0.044065       26.917408


## 2. Agent Run

The agent receives the priority ranking and autonomously decides which turbines to inspect,
what tools to call, and what maintenance action to recommend.

In [4]:
# ===============================
# RUN AGENT
# ===============================

print(f"Running WindOps agent (top_n={TOP_N})...\n")
action_plans, trace = run_agent(priority, df_risk, top_n=TOP_N)
print(f"Done. {len(action_plans)} action plan(s) generated.")

Running WindOps agent (top_n=3)...



TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

## 3. Agent Trace

Full step-by-step trace of what the agent did: which tools it called, what data it received,
and what it concluded before submitting each action plan.

In [6]:
# ===============================
# DISPLAY TRACE
# ===============================

STEP_LABELS = {
    "tool_call": "→ TOOL CALL",
    "tool_result": "← TOOL RESULT",
    "ai_message": "◆ AGENT",
}

STEP_COLORS = {
    "tool_call": "\033[94m",    # blue
    "tool_result": "\033[92m",  # green
    "ai_message": "\033[93m",   # yellow
}
RESET = "\033[0m"

print("=" * 72)
print("AGENT TRACE")
print("=" * 72)

for i, step in enumerate(trace, 1):
    kind = step.get("step", "unknown")
    label = STEP_LABELS.get(kind, kind.upper())
    color = STEP_COLORS.get(kind, "")

    print(f"\n{color}[{i:02d}] {label}{RESET}")

    if kind == "tool_call":
        print(f"     Tool  : {step['tool']}")
        args = step.get("input", {})
        for k, v in args.items():
            print(f"     {k:<12}: {v}")

    elif kind == "tool_result":
        content = step.get("content", "")
        # Try to pretty-print JSON
        try:
            import json
            parsed = json.loads(content)
            if isinstance(parsed, list):
                for item in parsed[:3]:
                    print(f"     {item}")
            else:
                for k, v in parsed.items():
                    print(f"     {k:<20}: {v}")
        except Exception:
            print(f"     {content[:300]}")

    elif kind == "ai_message":
        content = step.get("content", "")
        if content.strip():
            # Wrap at 68 chars
            words = content.split()
            line = "     "
            for word in words:
                if len(line) + len(word) + 1 > 72:
                    print(line)
                    line = "     " + word + " "
                else:
                    line += word + " "
            if line.strip():
                print(line)

print("\n" + "=" * 72)

AGENT TRACE


NameError: name 'trace' is not defined

## 4. Action Plans

Structured output from the agent — one plan per turbine, ready to be displayed
in the Streamlit frontend or exported via `src/io.py`.

In [ ]:
# ===============================
# DISPLAY ACTION PLANS
# ===============================

URGENCY_COLORS = {"high": "tomato", "medium": "orange", "low": "steelblue"}
URGENCY_ICONS  = {"high": "🔴", "medium": "🟡", "low": "🟢"}

if not action_plans:
    print("No action plans were generated.")
else:
    print(f"\n{'='*72}")
    print(f"ACTION PLANS — {SCENARIO.upper()} SCENARIO")
    print(f"{'='*72}")

    for plan in action_plans:
        icon    = URGENCY_ICONS.get(plan.get("urgency", "low"), "⚪")
        urgency = plan.get("urgency", "unknown").upper()
        tid     = plan.get("turbine_id", "?")
        fault   = plan.get("fault_hypothesis", "unknown")
        action  = plan.get("recommended_action", "")
        rationale = plan.get("rationale", "")

        print(f"\n{icon}  {tid}   [{urgency}]")
        print(f"   Fault hypothesis : {fault}")
        print(f"   Action           : {action}")
        print(f"   Rationale        : {rationale}")
        print(f"   {'─'*64}")

## 5. Priority vs Agent Coverage

Visual check: are the turbines the agent analysed the ones with highest risk?

In [ ]:
# ===============================
# PRIORITY VS AGENT COVERAGE
# ===============================

analysed_ids = {p["turbine_id"] for p in action_plans}
urgency_map  = {p["turbine_id"]: p.get("urgency", "low") for p in action_plans}

fig, ax = plt.subplots(figsize=(13, 4))

bar_colors = []
for _, row in priority.iterrows():
    tid = row["turbine_id"]
    if tid in analysed_ids:
        bar_colors.append(URGENCY_COLORS.get(urgency_map.get(tid, "low"), "steelblue"))
    else:
        bar_colors.append("lightgrey")

ax.bar(priority["turbine_id"], priority["priority_score"], color=bar_colors, edgecolor="white")

legend_patches = [
    mpatches.Patch(color="tomato",    label="Analysed — high urgency"),
    mpatches.Patch(color="orange",    label="Analysed — medium urgency"),
    mpatches.Patch(color="steelblue", label="Analysed — low urgency"),
    mpatches.Patch(color="lightgrey", label="Not analysed"),
]
ax.legend(handles=legend_patches, fontsize=8)
ax.set_ylabel("Priority score")
ax.set_title(f"Priority ranking with agent coverage — {SCENARIO} scenario")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 6. Notes on Traceability

The agent trace above shows every decision the model made:

- **TOOL CALL** — the agent chose to call a tool and with what arguments
- **TOOL RESULT** — what data the tool returned
- **AGENT** — intermediate reasoning before the next tool call

This traceability is a core feature of WindOps Copilot: operators can audit
not just *what* was recommended, but *why* and *based on which data*.

In the Streamlit frontend (notebook 04), this trace will be rendered as a
collapsible panel alongside each action plan.